<a href="https://colab.research.google.com/github/abhay-ugale-25/cinevibe/blob/main/cinevibe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install langchain-google-genai langchain-community pinecone sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.9/745.9 kB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Projects/cinevibe/data_raw/tmdb_5000_movies.csv")

In [ ]:
df_movies = df.drop(columns = ['budget', 'homepage', 'original_language', 'popularity', 'keywords','production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'title', 'vote_average', 'vote_count'])

df_movies = df_movies.dropna(axis = 0)

df_movies['combined_info'] = df_movies['original_title'].astype(str) + ":"+ df_movies['overview'].astype(str)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

model = "sentence-transformers/all-MiniLM-L6-v2"

model_kwargs = {'device': 'cuda'}

embed_model = HuggingFaceEmbeddings(
    model_name=model,
    model_kwargs=model_kwargs
)

/tmp/ipython-input-2750696390.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key = userdata.get('pinecone'))

In [ ]:
index_name = 'movies-index'
index = pc.Index(index_name)

In [ ]:
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '150',
                                    'content-type': 'application/json',
                                    'date': 'Tue, 10 Feb 2026 14:26:01 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '2',
                                    'x-pinecone-request-latency-ms': '1',
                                    'x-pinecone-response-duration-ms': '5'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}

In [ ]:
from tqdm.auto import tqdm

batch_size = 100

for i in tqdm(range(0, len(df_movies), batch_size), desc = "Upserting Batches"):
  batch = df_movies.iloc[i:i+batch_size]

  texts_to_embeds = batch['combined_info'].tolist()

  vectors = embed_model.embed_documents(texts_to_embeds)

  metadatas = []

  for ind, row in batch.iterrows():
    meta = {
        "title": row['original_title'],
        "genres": row['genres'],
        "overview": row['overview'],

    }
    metadatas.append(meta)

  ids = batch['id'].astype(str).to_list()

  to_upsert = list(zip(ids, vectors, metadatas))

  index.upsert(vectors=to_upsert)

Upserting Batches:   0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
import os
import logging
import warnings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

class CineVibeEngine:
  def __init__(self, pinecone_key, google_key, index_name = 'movies-index'):
    print("Initializing CineVibe")

    self.embed_model = HuggingFaceEmbeddings(
        model_name = "sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs =  {'device': 'cuda'}
    )

    print("Embedding model loaded!")

    self.pc = Pinecone(api_key = pinecone_key)
    self.index = self.pc.Index(index_name)

    print(f"Connected to Pinecone Index: {index_name}")

    self.llm = ChatGoogleGenerativeAI(
              model = 'gemini-2.5-flash',
              google_api_key = userdata.get('GoogleAIStudio'),
              temperature=0.7
              )

    print("Gemini AI loaded!")

  def search(self, query, top_k = 5):
    query_vector = self.embed_model.embed_query(query)

    results = self.index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )

    return results

  def format_context(self, results):
    context_string = ""
    for idx, match in enumerate(results['matches'], start = 1):
      title = match['metadata']['title']
      overview = match['metadata']['overview']
      genres = match['metadata']['genres']

      context_string += f"{idx}. Title: {title}\n"
      context_string += f"       Genres: {genres}\n"
      context_string += f"       Overview: {overview}\n\n"

    return context_string

  def create_prompt(self, user_query, context):
    prompt = f"""
        You are a movie expert. The user wants a movie with this vibe: "{user_query}".

        Here are the top 5 matches from our database:
        {context}

        Task:
        1. Pick the ONE best movie from this list.
        2. Explain why it fits the user's mood creatively.
        3. Suggest 2 alternatives.
        4. Use emojis.
        5. ALign the output in attracive way.
        """
    return prompt

  def recommend(self, query):
    if not query:
      return "Describe your vibe!"

    search_result = self.search(query)

    context = self.format_context(search_result)

    prompt = self.create_prompt(query, context)

    response = self.llm.invoke(prompt)

    return response.content


In [ ]:
cinevibe = CineVibeEngine(
    pinecone_key = userdata.get('pinecone'),
    google_key = userdata.get('GoogleAIStudio')
)

print("Welcome To CineVibe!")
print("powered by Pinecone and Gemini 2.5 Flash")

while True:
  try:
    # User input
    user_query = input("Your Vibe: ")

    if user_query.lower() in ['exit', 'quit']:
      print("Shutting down CineVibe.")
      break

    print("Processing...")

    result = cinevibe.recommend(user_query)

    print(f"\n{result}\n")
    print("-"*100)

  except KeyboardInterrupt:
    print("Force Stopping...!")
    break
  except Exception as e:
    print(f"An error occurred: {e}")
    break

Initializing CineVibe


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!
Connected to Pinecone Index: movies-index
Gemini AI loaded!
WELCOME TO CINEVIBE!
powered by Pinecone and Gemini 2.5 Flash
Your Vibe: Heartwarming comedy
Processing...

Here's your perfect pick for a heartwarming comedy! 🎬🍿

### ✨ Your Top Pick for a Heartwarming Comedy ✨

**The Good Heart** 💖

This isn't just a movie; it's a warm, unexpected embrace in the most unlikely of places. Imagine a gruff, world-weary bar owner, Jacques, whose heart seems as hardened as the New York streets outside his dive. Then, along comes Lucas, a lost soul who needs a home. "The Good Heart" masterfully weaves a tale of an unlikely mentorship, where a curmudgeon finds a purpose in nurturing a young man, teaching him the ropes of life (and bartending). It’s about finding family where you least expect it, proving that even the most guarded hearts can open up and share warmth. You'll laugh, you might shed a tear, and you'll definitely leave with a fuzzy feeling, knowing that connection 